<a href="https://colab.research.google.com/github/heisdenverr/llm-interpretability/blob/master-branch/SAE_of_Qwen2_0.5B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2-0.5B', dtype=torch.float32)
model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2-0.5B', dtype=torch.float32)

messages = ["The Eiffel Tower is located in"
]

inputs = tokenizer(messages, return_tensors='pt')

outputs = model(**inputs)
print(outputs.logits.shape)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

torch.Size([1, 8, 151936])


torch.Size([1, 26, 151936])-->(B, N_TOKENS, VOCAB_SIZE)


In [ ]:
last_token = outputs.logits[0, -1,:]
token_id = torch.argmax(last_token)
decode = tokenizer.decode(token_id)
print(f"Token id: {token_id}")
print(f"Decoded to text: {decode}")

Token id: 12095
Decoded to text:  Paris


In [ ]:
tokenizer.decode(torch.argmax(outputs.logits, dim=-1))

[' following-el Tower is the in Paris']

In [ ]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [ ]:
model.model.layers[12].mlp

Qwen2MLP(
  (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
  (up_proj): Linear(in_features=896, out_features=4864, bias=False)
  (down_proj): Linear(in_features=4864, out_features=896, bias=False)
  (act_fn): SiLUActivation()
)

In [ ]:
captured = {}
layers = [9, 10, 11, 12, 13, 14]

def get_hook(layer_id):
  def hook(module, input, output):
    captured[layer_id] = output.cpu().detach()
  return hook

handles = [model.model.layers[n].mlp.register_forward_hook(get_hook(n)) for n in layers]


In [ ]:
from datasets import load_dataset
ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:500]")

In [ ]:
text = [text for text in ds['text'] if text.strip()]
len(text)

338

In [ ]:
store= {9: [],
        10: [],
        11: [],
        12: [],
        13: [],
        14 :[]
           }

for i in text:
  tokenids = tokenizer(i,
                      return_tensors='pt',
                      truncation=True,
                      max_length=128)
  with torch.no_grad():
    y_pred = model(**tokenids)

  for layer in store:
    store[layer].append(captured[layer])

for handle in handles:
  handle.remove()

In [ ]:
len(store[12])

338

In [ ]:
store[12][0].shape

torch.Size([1, 8, 896])

In [ ]:
activation_9 = torch.cat([t.squeeze(0) for t in store[9]], dim=0)
activation_10 = torch.cat([t.squeeze(0) for t in store[10]], dim=0)
activation_11 = torch.cat([t.squeeze(0) for t in store[11]], dim=0)
activation_12 = torch.cat([t.squeeze(0) for t in store[12]], dim=0)
activation_13 = torch.cat([t.squeeze(0) for t in store[13]], dim=0)
activation_14 = torch.cat([t.squeeze(0) for t in store[14]], dim=0)

In [ ]:
activation_10.shape

torch.Size([18639, 896])

In [ ]:
from torch import nn

class SAE(nn.Module):

  def __init__(self, in_dim, out_dim):
    super().__init__()
    self.encoder = nn.Linear(in_features=in_dim, out_features=out_dim)
    self.decoder = nn.Linear(in_features=out_dim, out_features=in_dim)
    self.relu = nn.ReLU()

  def forward(self, x: torch.Tensor):

    hidden = self.relu(self.encoder(x))
    reconstruction = self.decoder(hidden)
    return reconstruction, hidden

In [ ]:
sparsity = SAE(in_dim=896, out_dim=3584)
sparsity

SAE(
  (encoder): Linear(in_features=896, out_features=3584, bias=True)
  (decoder): Linear(in_features=3584, out_features=896, bias=True)
  (relu): ReLU()
)

In [ ]:
sparsity(activation_9)

(tensor([[ 0.1062, -0.0413, -0.4470,  ..., -0.0113,  0.1753, -0.1350],
         [ 0.2217, -0.0190, -0.2152,  ..., -0.2472,  0.0129, -0.2193],
         [ 0.0167, -0.2077, -0.1304,  ..., -0.0165,  0.0657, -0.1014],
         ...,
         [ 0.1162, -0.0501,  0.1964,  ...,  0.0037,  0.0811, -0.1451],
         [ 0.1943, -0.0536, -0.0357,  ...,  0.1702, -0.0023,  0.0485],
         [ 0.0856,  0.0458, -0.1269,  ..., -0.0063,  0.0236, -0.0760]],
        grad_fn=<AddmmBackward0>),
 tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0507, 0.0283, 0.0453,  ..., 0.2137, 0.1432, 0.0270],
         [0.0000, 0.0459, 0.0212,  ..., 0.0049, 0.0283, 0.0055],
         ...,
         [0.1223, 0.0295, 0.0900,  ..., 0.0764, 0.0933, 0.0468],
         [0.1168, 0.1181, 0.0458,  ..., 0.1624, 0.0000, 0.1471],
         [0.0484, 0.0000, 0.0695,  ..., 0.0263, 0.0808, 0.0292]],
        grad_fn=<ReluBackward0>))

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
train_dataloader = DataLoader(activation_9, batch_size=256)
tok = next(iter(train_dataloader))
tok.shape

torch.Size([256, 896])

In [ ]:
tok.shape

torch.Size([256, 896])

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(sparsity.parameters(), lr=1e-4)
lambda_val  = 5e-1
for epoch in range(40):
  total_loss = 0
  for x in train_dataloader:
    optimizer.zero_grad()
    reconstrunction, hidden = sparsity(x)
    loss = loss_fn(reconstrunction, x) + lambda_val * hidden.abs().mean()
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  print(f"Epoch {epoch+1} loss: {total_loss / len(train_dataloader):.4f}")

Epoch 1 loss: 0.0118
Epoch 2 loss: 0.0113
Epoch 3 loss: 0.0108
Epoch 4 loss: 0.0104
Epoch 5 loss: 0.0101
Epoch 6 loss: 0.0098
Epoch 7 loss: 0.0095
Epoch 8 loss: 0.0093
Epoch 9 loss: 0.0090
Epoch 10 loss: 0.0088
Epoch 11 loss: 0.0086
Epoch 12 loss: 0.0084
Epoch 13 loss: 0.0082
Epoch 14 loss: 0.0081
Epoch 15 loss: 0.0079
Epoch 16 loss: 0.0078
Epoch 17 loss: 0.0077
Epoch 18 loss: 0.0075
Epoch 19 loss: 0.0074
Epoch 20 loss: 0.0073
Epoch 21 loss: 0.0072
Epoch 22 loss: 0.0071
Epoch 23 loss: 0.0070
Epoch 24 loss: 0.0069
Epoch 25 loss: 0.0068
Epoch 26 loss: 0.0067
Epoch 27 loss: 0.0066
Epoch 28 loss: 0.0066
Epoch 29 loss: 0.0065
Epoch 30 loss: 0.0064
Epoch 31 loss: 0.0063
Epoch 32 loss: 0.0063
Epoch 33 loss: 0.0062
Epoch 34 loss: 0.0061
Epoch 35 loss: 0.0061
Epoch 36 loss: 0.0060
Epoch 37 loss: 0.0060
Epoch 38 loss: 0.0059
Epoch 39 loss: 0.0058
Epoch 40 loss: 0.0058


In [ ]:

with torch.no_grad():
  batch = next(iter(train_dataloader))
  _, hidden = sparsity(batch)
  sparsity_ratio = (hidden == 0).float().mean()
  print(f"Sparsity: {sparsity_ratio:.4f}")

Sparsity: 0.8663


In [ ]:
with torch.no_grad():
  _, hidden = sparsity(activation_9)
n_0 = hidden[:, 0]
top10 = torch.topk(n_0, 10)
print(top10.indices)

tensor([12729, 12730, 12765, 12761, 12472, 13656, 12724, 13654, 12766, 13653])


In [ ]:
lengths = torch.tensor([t.shape[1] for t in store[9]])

In [ ]:
boundaries = torch.cumsum(lengths, dim=0)
print(boundaries[:10])

tensor([  8, 136, 246, 362, 368, 496, 624, 715, 721, 849])


In [ ]:
top10

torch.return_types.topk(
values=tensor([0.6811, 0.6297, 0.6265, 0.5415, 0.4999, 0.4978, 0.4796, 0.4559, 0.4455,
        0.4344]),
indices=tensor([12729, 12730, 12765, 12761, 12472, 13656, 12724, 13654, 12766, 13653]))

In [ ]:
index = []
for i in top10.indices:
  index.append((boundaries>i).nonzero()[0])
index

[tensor([233]),
 tensor([233]),
 tensor([234]),
 tensor([234]),
 tensor([230]),
 tensor([243]),
 tensor([233]),
 tensor([243]),
 tensor([234]),
 tensor([243])]

In [ ]:
for n in index:
  print(text[n.item()])

 In the off @-@ season the Blue Jackets ' approach to building their team changed , moving from a team of young developing players into one with established players . The first deal General Manager Scott Howson made was the acquisition of All @-@ Star forward Jeff Carter on June 23 , 2011 . The deal sent Jakub Voracek , Columbus ' first @-@ round draft choice , the eighth overall , and their third @-@ round pick in the 2011 Draft to the Philadelphia Flyers in exchange for Carter . The trade received a positive response in Columbus from fans and management who felt they finally had a number one center to play alongside of their best player , Rick Nash . Next , they traded for the negotiating rights of soon to be free agent James Wisniewski . Wisniewski scored a career high 51 points during the 2010 – 11 season , splitting time between the New York Islanders and Montreal Canadiens . The point total was fifth @-@ highest in the league for defenseman scoring , tying Tobias Enstrom . The Bl

[233, 233, 234, 234, 230, 243, 233, 243, 234, 243]

In [ ]:
sd_idx = []
idx = []
for neuron in hidden:
  t3 = torch.topk(neuron, 3)
  idx.append([n.item() for n in t3.indices])

  sd = [(boundaries>i).nonzero()[0] for i in t3.indices]
  sd_idx.append(tuple(n.item() for n in sd))

result = dict(zip(sd_idx, idx))
result

{(6, 19, 41): [538, 1643, 3428],
 (2, 7, 3): [212, 678, 246],
 (23, 1, 3): [1971, 16, 251],
 (6, 1, 11): [574, 115, 1057],
 (11, 2, 7): [1034, 212, 678],
 (1, 40, 11): [43, 3338, 1100],
 (22, 40, 37): [1915, 3365, 3085],
 (22, 37, 35): [1915, 3177, 2953],
 (7, 2, 40): [678, 212, 3366],
 (7, 23, 35): [711, 2093, 2993],
 (7, 22, 37): [712, 1946, 3147],
 (23, 26, 38): [1988, 2253, 3201],
 (1, 16, 40): [73, 1438, 3327],
 (6, 30, 23): [544, 2565, 2051],
 (9, 2, 10): [723, 207, 915],
 (31, 40, 41): [2683, 3417, 3473],
 (10, 32, 41): [950, 2757, 3484],
 (42, 10, 35): [3556, 954, 3047],
 (2, 38, 42): [229, 3250, 3580],
 (1, 7, 29): [118, 651, 2531],
 (18, 24, 38): [1545, 2179, 3233],
 (41, 9, 23): [3505, 783, 2014],
 (6, 14, 28): [568, 1279, 2365],
 (23, 29, 11): [1988, 2420, 990],
 (38, 14, 32): [3247, 1313, 2812],
 (2, 23, 35): [148, 2077, 3027],
 (2, 30, 18): [229, 2636, 1545],
 (4, 38, 23): [364, 3211, 2027],
 (2, 37, 2): [177, 3120, 241],
 (16, 37, 2): [1412, 3098, 206],
 (40, 21, 2): [33

In [ ]:
tokenizer.decode([678])

' all'

In [ ]:
sentence_idx = (boundaries > 1988).nonzero()[0].item()
print(text[sentence_idx])

 PlayStation Official Magazine - UK praised the story 's blurring of Gallia 's moral standing , art style , and most points about its gameplay , positively noting the latter for both its continued quality and the tweaks to balance and content . Its one major criticism were multiple difficulty spikes , something that had affected the previous games . Heath Hindman of gaming website PlayStation Lifestyle praised the addition of non @-@ linear elements and improvements or removal of mechanics from Valkyria Chronicles II in addition to praising the returning gameplay style of previous games . He also positively noted the story 's serious tone . Points criticized in the review were recycled elements , awkward cutscenes that seemed to include all characters in a scene for no good reason , pacing issues , and occasional problems with the game 's AI . 



In [ ]:
from collections import Counter

all_sentences = [idx for key in result.keys() for idx in key]
cm = Counter(all_sentences).most_common(10)

In [ ]:
r_s = ''
for i in cm:
  print(text[i[0]])


 The game takes place during the Second Europan War . Gallian Army Squad 422 , also known as " The Nameless " , are a penal military unit composed of criminals , foreign deserters , and military offenders whose real names are erased from the records and thereon officially referred to by numbers . Ordered by the Gallian military to perform the most dangerous missions that the Regular Army and Militia will not do , they are nevertheless up to the task , exemplified by their motto , Altaha Abilia , meaning " Always Ready . " The three main characters are No.7 Kurt Irving , an army officer falsely accused of treason who wishes to redeem himself ; Ace No.1 Imca , a female Darcsen heavy weapons specialist who seeks revenge against the Valkyria who destroyed her home ; and No.13 Riela Marcellis , a seemingly jinxed young woman who is unknowingly a descendant of the Valkyria . Together with their fellow squad members , these three are tasked to fight against a mysterious Imperial unit known as

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
r_s

' The game takes place during the Second Europan War . Gallian Army Squad 422 , also known as " The Nameless " , are a penal military unit composed of criminals , foreign deserters , and military offenders whose real names are erased from the records and thereon officially referred to by numbers . Ordered by the Gallian military to perform the most dangerous missions that the Regular Army and Militia will not do , they are nevertheless up to the task , exemplified by their motto , Altaha Abilia , meaning " Always Ready . " The three main characters are No.7 Kurt Irving , an army officer falsely accused of treason who wishes to redeem himself ; Ace No.1 Imca , a female Darcsen heavy weapons specialist who seeks revenge against the Valkyria who destroyed her home ; and No.13 Riela Marcellis , a seemingly jinxed young woman who is unknowingly a descendant of the Valkyria . Together with their fellow squad members , these three are tasked to fight against a mysterious Imperial unit known a